<div style="display: flex; justify-content: space-between; align-items: flex-start; border-bottom: 2px solid #555555; padding-bottom: 15px; margin-bottom: 12px;">
    <div style="width: 100%;">
        <h2>
            <span style="color: #B30033;">▍</span>CARDIS
        </h2>
        <h1 style="margin-top: -10px;">
            Equidad y Calibración (Post-Modelado)
        </h1>
    </div>
</div>

**Notebook de Experimentación**

**Proyecto:** Clinical AI Risk Decision Support System (CARDIS)  
**Asignatura:** Desarrollo e Integración de Servicios de IA (DISIA)  
**Grupo 6:** Alonso Crespo · Diego Guerrero · José Lara · Cristina Serrano  
**Fecha:** Marzo 2026  

### Objetivo

Este notebook verifica si el modelo obtenido en pasos anteriores es equitativo y posee un rendimiento similar para cualquier franja de edad, y si las predicciones hechas por el modelo encajan con el riesgo real.

### Research Questions abordadas

- **RQ1:** ¿El modelo posee un rendimiento similar sea cual sea el rango de edad?
- **RQ2:** ¿Las predicciones del modelo se traducen en resultados reales?

## 0. Configuración del entorno

In [ ]:
# ── Librerías ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# ML — Métricas de evaluación y calibración
from sklearn.metrics import (
    f1_score, recall_score, precision_score, 
    brier_score_loss, confusion_matrix, classification_report
)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.base import BaseEstimator, TransformerMixin

# Librería de equidad
from fairlearn.metrics import MetricFrame, equalized_odds_difference, selection_rate

# Configuración de reproducibilidad
SEED = 42
np.random.seed(SEED)

print("Entorno configurado correctamente.")
print(f"  NumPy       : {np.__version__}")
print(f"  pandas      : {pd.__version__}")
print(f"  scikit-learn: {__import__('sklearn').__version__}")
print(f"  Fairlearn   : {__import__('fairlearn').__version__}")
print(f"  SEED        : {SEED}")

Entorno configurado correctamente.
  NumPy       : 2.2.6
  pandas      : 2.3.3
  scikit-learn: 1.7.2
  Fairlearn   : 0.13.0
  SEED        : 42


## 1. Carga de datos y preparación

En primer lugar, cargaremos los datos originales que almacenamos en hitos anteriores. Por otro lado, los datos procesados provienen del pipeline de preprocesamiento del Hito 2 (`cardis_pipeline.ipynb`), que exportó dos versiones del dataset de entrenamiento:

| Archivo | Contenido | Uso |
|---------|-----------|-----|
| `cardio_risk_train_fe.csv` | Solo Feature Engineering (sin escalar) | Modelos de árboles (RF, LightGBM, XGBoost) |
| `cardio_risk_train_pipeline.csv` | FE + imputación + Target Encoding + RobustScaler | Regresión Logística |

Aunque esto se menciona en el notebook de (`cardis_modelado.ipynb`), la razón de tener dos conjuntos de datos distintos es porque el modelo requiere que los datos estén en escalas similares. Por ello, utilizaremos el segundo dataset.

Por otra parte, se utilizará el modelo obtenido tras entrenar una instancia de LightGBM (`cardis_modelo_final.joblib`), almacenado en la carpeta _models_, y obtenido en el cuaderno mendionado anteriormente.

In [25]:
# ── Target Encoder (replicado de cardis_pipeline.ipynb) ────────────────────────
# Sustituye cada categoría por la media suavizada del target en esa categoría.
# El smoothing evita sobreajuste en categorías con pocas observaciones.
class TargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, columns, smoothing=1.0):
        self.columns = columns
        self.smoothing = smoothing
        self.mapping_ = {}
        self.global_mean_ = None

    def fit(self, X, y):
        X = X.copy()
        self.global_mean_ = y.mean()
        for col in self.columns:
            if col in X.columns:
                stats = y.groupby(X[col]).agg(['mean', 'count'])
                # Fórmula de suavizado: pondera entre la media de la categoría
                # y la media global según el nº de observaciones
                smooth = (
                    (stats['count'] * stats['mean'] + self.smoothing * self.global_mean_)
                    / (stats['count'] + self.smoothing)
                )
                self.mapping_[col] = smooth.to_dict()
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.columns:
            if col in X.columns:
                # Categorías no vistas en fit se rellenan con la media global
                X[col] = X[col].map(self.mapping_.get(col, {})).fillna(self.global_mean_)
        return X


try:
    # Carga del dataset original
    df_original = pd.read_csv(Path("../data/raw/cardio_risk_train.csv"))

    # Carga del dataset escalado
    df_escalado = pd.read_csv(Path("../data/processed/cardio_risk_train_pipeline.csv"))
    
    # Carga del modelo entrenado
    model = joblib.load(Path("models/cardis_modelo_final.joblib"))
    
    print("Recursos cargados correctamente:")
    print(f"   Dataset original: {df_original.shape[0]} filas y {df_original.shape[1]} columnas.")
    print(f"   Dataset escalado: {df_escalado.shape[0]} filas y {df_escalado.shape[1]} columnas.")

    if 'pipeline' in model:
        pipeline_final = model['pipeline']

    print(f"   Modelo : {type(pipeline_final).__name__}")
except FileNotFoundError as e:
    print(f"Error: No se encontró el archivo. {e}")

Recursos cargados correctamente:
   Dataset original: 24000 filas y 21 columnas.
   Dataset escalado: 24000 filas y 39 columnas.
   Modelo : LGBMClassifier


## 2. Generación de predicciones y probabilidades

Antes de empezar a trabajar con el análisis de los sesgos, se deben conocer los resultados del modelo tras hacer inferencia con los datos originales. Para ello, utilizaremos el conjunto de datos escalado, ya que son los apropiados para usar nuestro modelo. Utilizaremos un umbral dinámico sobre los percentiles para que el modelo sea lo más preciso posible.

In [29]:
# 1. Preparación de las características y la columna objetivo
X_val = df_escalado.drop(columns=['riesgo_cv'])
y_true = df_escalado['riesgo_cv']

# 2. Obtener probabilidades del modelo
lgbm_model = model['modelo']
y_proba = lgbm_model.predict_proba(X_val)[:, 1]

# 3. Ajuste de umbral dinámico
umbral_dinamico = np.percentile(y_proba, 78)
y_pred = (y_proba >= umbral_dinamico).astype(int)

# 4. Reporte de resultados
casos_reales = int(y_true.sum())
casos_predichos = int(y_pred.sum())

print(f"   Generación de predicciones completada")
print(f"   Casos positivos REALES    : {casos_reales}")
print(f"   Casos positivos PREDICHOS : {casos_predichos}")
print(f"   Diferencia absoluta       : {abs(casos_reales - casos_predichos)}")
print("-" * 50)
print("   Reporte de rendimiento")
print(classification_report(y_true, y_pred))

   Generación de predicciones completada
   Casos positivos REALES    : 5256
   Casos positivos PREDICHOS : 5280
   Diferencia absoluta       : 24
--------------------------------------------------
   Reporte de rendimiento
              precision    recall  f1-score   support

           0       0.84      0.84      0.84     18744
           1       0.44      0.45      0.45      5256

    accuracy                           0.76     24000
   macro avg       0.64      0.65      0.64     24000
weighted avg       0.76      0.76      0.76     24000



Al trabajar con un umbral dinámico, estamos forzando al modelo a estar igual de equilibrado como en la realidad, donde existen en torno al 22% de casos positivos. De esta forma, operaciones posteriores para analizar la equidad y calibración del modelo se nos son facilitados.

## 3. Partición del conjunto de datos por franja de edad

Con el fin de analizar los posibles sesgos del modelo obtenido para pacientes en diferentes franjas de edad, vamos a particionar el dataset en 5 franjas diferentes:
- 30 a 39 años
- 40 a 49 años
- 50 a 59 años
- 60 a 69 años
- 70 a 84 años

Como podemos observar, la última partición incluye hasta los 84 años, edad máxima presente en nuestro conjunto de datos.

In [33]:
# 1. Definición de cortes
bins = [30, 40, 50, 60, 70, 85]
labels = ['30-39', '40-49', '50-59', '60-69', '70-84']

# 2. Creación del DataFrame de evaluación
df_eval = pd.DataFrame({
    'y_true': y_true,
    'y_pred': y_pred,
    'y_proba': y_proba,
    'edad_real': df_original['edad']
})

# 3. Aplicar el binning
df_eval['franja_edad'] = pd.cut(
    df_eval['edad_real'], 
    bins=bins, 
    labels=labels, 
    right=False
)

# 4. Resumen de la distribución
print("Distribución por décadas:")
resumen_bins = df_eval.groupby('franja_edad', observed=True).agg(
    Total_Pacientes=('y_true', 'count'),
    Casos_Reales=('y_true', 'sum'),
    Porcentaje_Positivos=('y_true', lambda x: (x.mean() * 100).round(2))
)

display(resumen_bins)

Distribución por décadas:


,Total_Pacientes,Casos_Reales,Porcentaje_Positivos
franja_edad,,,
30-39,4353,138,3.17
40-49,4487,284,6.33
50-59,4340,614,14.15
60-69,4308,1173,27.23
70-84,6512,3047,46.79


Como cabría esperar, el porcentaje de casos positivos aumenta en cada franja de edad, indicando una correlación lineal directa.

## 7. Exportación del modelo final y predicciones

Se exportan todos los artefactos necesarios para el equipo:

- **`cardis_modelo_final.joblib`**: contiene el modelo LightGBM entrenado, el Target Encoder, el imputer, el threshold óptimo, y los nombres de las features. Esto permite cargar el modelo en un solo paso para SHAP y para el despliegue.
- **`predicciones_lgbm_cv.csv`**: predicciones por fold de CV (y_true, y_pred, y_proba), necesarias para el análisis de equidad y calibración.
- **`predicciones_test.csv`**: predicciones sobre el dataset de evaluación sin etiquetar.
- **`hiperparametros_optimos.json`**: parámetros de los 4 modelos para reproducibilidad.


In [ ]:
# ── Entrenamiento final sobre TODO el training set ────────────────────────────
# (sin hold-out, para maximizar los datos disponibles en el modelo final)
print("Entrenando modelo final sobre el training set completo...")

te_export = TargetEncoder(columns=cols_alta_card_fe, smoothing=1.0)
te_export.fit(X_fe, y)
X_fe_encoded = te_export.transform(X_fe)

imp_export = SimpleImputer(strategy='median')
X_fe_imputed = pd.DataFrame(
    imp_export.fit_transform(X_fe_encoded),
    columns=X_fe_encoded.columns
)

modelo_final = lgb.LGBMClassifier(
    n_estimators=params_lgbm['n_estimators'],
    learning_rate=params_lgbm['learning_rate'],
    max_depth=params_lgbm['max_depth'],
    num_leaves=params_lgbm['num_leaves'],
    min_child_samples=params_lgbm['min_child_samples'],
    reg_alpha=params_lgbm['reg_alpha'],
    reg_lambda=params_lgbm['reg_lambda'],
    subsample=params_lgbm['subsample'],
    colsample_bytree=params_lgbm['colsample_bytree'],
    scale_pos_weight=spw_lgbm,
    random_state=SEED, verbose=-1, n_jobs=-1
)
modelo_final.fit(X_fe_imputed, y)

print(f"  ✓ Modelo: LightGBM")
print(f"  ✓ Threshold óptimo: {mejor_thr:.2f}")
print(f"  ✓ Features: {X_fe_imputed.shape[1]}")

# ── Exportar artefactos del modelo ────────────────────────────────────────────
artefactos = {
    'modelo': modelo_final,
    'target_encoder': te_export,
    'imputer': imp_export,
    'threshold': mejor_thr,
    'feature_names': X_fe_imputed.columns.tolist(),
    'params': params_lgbm,
    'seed': SEED
}
joblib.dump(artefactos, 'models/cardis_modelo_final.joblib')
print(f"\n→ Modelo exportado: models/cardis_modelo_final.joblib")

# ── Exportar predicciones por fold ───────────────────────────
res_lgbm['predicciones'].to_csv('outputs/predicciones_lgbm_cv.csv', index=False)
print(f"→ Predicciones CV: outputs/predicciones_lgbm_cv.csv")

# ── Exportar hiperparámetros de los 4 modelos ─────────────────────────────────
all_params = {
    'logistic_regression': params_lr,
    'random_forest': params_rf,
    'lightgbm': params_lgbm,
    'xgboost': params_xgb,
    'threshold_optimo': float(mejor_thr)
}
with open('outputs/hiperparametros_optimos.json', 'w') as f:
    json.dump(all_params, f, indent=2, default=str)
print(f"→ Hiperparámetros: outputs/hiperparametros_optimos.json")

# ── Predicciones sobre el test set (sin etiquetar) ────────────────────────────
X_test_encoded = te_export.transform(df_test_fe)
X_test_imputed = pd.DataFrame(
    imp_export.transform(X_test_encoded),
    columns=X_test_encoded.columns
)
y_test_proba = modelo_final.predict_proba(X_test_imputed)[:, 1]
y_test_pred = (y_test_proba >= mejor_thr).astype(int)

df_test_pred = pd.DataFrame({
    'y_pred': y_test_pred,
    'y_proba': y_test_proba
})
df_test_pred.to_csv('outputs/predicciones_test.csv', index=False)
print(f"→ Predicciones test: outputs/predicciones_test.csv")
print(f"  Distribución: {y_test_pred.sum()} alto riesgo ({y_test_pred.mean():.2%})")


Entrenando modelo final sobre el training set completo...
  ✓ Modelo: LightGBM
  ✓ Threshold óptimo: 0.59
  ✓ Features: 38

→ Modelo exportado: models/cardis_modelo_final.joblib
→ Predicciones CV: outputs/predicciones_lgbm_cv.csv
→ Hiperparámetros: outputs/hiperparametros_optimos.json
→ Predicciones test: outputs/predicciones_test.csv
  Distribución: 3754 alto riesgo (23.46%)


La distribución de predicciones en el test set (23,5% alto riesgo) es coherente con la prevalencia real del dataset de entrenamiento (21,9%). El ligero exceso de predicciones positivas es esperable dado que el modelo usa `scale_pos_weight = 3,56`, que sesga las probabilidades hacia la clase minoritaria. El threshold de 0,59 corrige parcialmente este efecto, pero la proporción final sigue siendo razonable y no presenta signos de sesgo extremo.

El modelo final, el Target Encoder, el imputer y el threshold quedan encapsulados en un único archivo `.joblib` para que puedan cargarse directamente en el despliegue y en el análisis SHAP sin necesidad de re-ejecutar el pipeline de entrenamiento.


## 8. Resumen de resultados

Los archivos generados por este notebook se resumen a continuación:

| Archivo | Contenido |
| :--- | :--- |
| `outputs/tabla_comparativa_modelos.csv` | Métricas de los 4 modelos (5-fold CV) |
| `outputs/tests_wilcoxon.csv` | Tests estadísticos entre pares |
| `figures/fig_ablation_study.png` | Contribución incremental del FE |
| `figures/fig_threshold_optimization.png` | Curva F1 vs threshold |
| `outputs/predicciones_lgbm_cv.csv` | y_true, y_pred, y_proba por fold |
| `models/cardis_modelo_final.joblib` | Modelo + TE + imputer + threshold |
| `outputs/hiperparametros_optimos.json` | Params de los 4 modelos + threshold |
| `outputs/predicciones_test.csv` | Predicciones en test set |
| `outputs/dependencias.json` | Versiones de librerías |

## 9. Conclusiones del experimento

### ¿Qué modelo hemos elegido y por qué?

Tras comparar cuatro modelos (desde una Regresión Logística clásica hasta algoritmos de *boosting* como LightGBM y XGBoost) el resultado más llamativo es lo **cerca que están todos entre sí**. La diferencia entre el mejor (LightGBM, F1 = 0,7197) y el *baseline* lineal (Regresión Logística, F1 = 0,7158) es de apenas 0,004 puntos: prácticamente indistinguible y, de hecho, no significativa estadísticamente. Esto nos dice algo importante: **el pipeline de datos que construimos en el Hito 2 es sólido**. Cuando modelos de familias muy diferentes convergen en rendimiento, la señal está en los datos, no en el algoritmo.

Seleccionamos **LightGBM** como modelo final, no porque sea drásticamente superior en F1, sino por el conjunto de sus propiedades: tiene la mejor calibración de probabilidades (Brier = 0,0959), es compatible con SHAP para generar las explicaciones que CARDIS necesita mostrar al médico, y su velocidad de inferencia (∼1 segundo) lo hace viable para despliegue en tiempo real.

### ¿Qué hemos aprendido?

Los resultados permiten responder a las dos preguntas de investigación planteadas:

**RQ1 - ¿Qué modelo ofrece el mejor equilibrio rendimiento–explicabilidad?** LightGBM, pero con un matiz relevante: la Regresión Logística, un modelo inherentemente interpretable, rinde casi igual. Esto refuerza el caso para la explicabilidad en CARDIS: no necesitamos sacrificar transparencia para obtener buen rendimiento.

**RQ2 - ¿Cuánto aporta la ingeniería de características?** Mucho. El *ablation study* muestra que pasar de las variables clínicas originales (F1 = 0,6321) al pipeline completo (F1 = 0,7110) supone una mejora de casi 8 puntos. Las variables que más aportan son las contextuales (actividad física, indicadores de *missing*, temporales), seguidas de las *keywords* clínicas extraídas de las notas médicas. El esfuerzo de preprocesamiento del Hito 2 ha merecido la pena.

### En resumen

El sistema CARDIS, con los datos disponibles, es capaz de **identificar correctamente a casi 8 de cada 10 pacientes de alto riesgo cardiovascular** (recall ≈ 0,78) con una tasa de falsas alarmas razonable (precision ≈ 0,67). No es un sistema perfecto (ninguno lo es con datos sintéticos y sin variables como el sexo) pero es un punto de partida sólido para un sistema de apoyo a la decisión clínica que, recordemos, no toma decisiones por sí solo: presenta una estimación de riesgo al médico, que es quien decide.

## Declaración de uso de inteligencia artificial

Los autores del presente trabajo declaran que todas las decisiones de diseño experimental, la selección de modelos y métricas, la interpretación de los resultados y las conclusiones han sido tomadas de forma íntegra por los miembros del equipo, apoyándose en los conocimientos adquiridos en la asignatura y en la literatura de referencia.

La inteligencia artificial (Claude, Anthropic) ha sido empleada como herramienta de apoyo en las siguientes tareas instrumentales:

- **Asistencia en código:** generación de estructuras base de notebooks, funciones de evaluación y visualización, que posteriormente fueron revisadas, adaptadas y validadas por los autores.
- **Redacción y estilo:** revisión ortográfica y estilística del texto, así como sugerencias de estructura para las celdas de markdown explicativas.
- **Apoyo en la resolución de problemas técnicos:** en particular, diagnóstico de problemas de convergencia en la optimización de la Regresión Logística (selección del solver `liblinear` frente a `saga`) y ajuste de espacios de hiperparámetros.
- **Contraste de interpretaciones:** en algunos resultados cuya lectura no era inmediata (como la aparente contradicción entre AUC-ROC y F1 en la Regresión Logística, o el perfil clínico de los falsos negativos), la IA se utilizó como interlocutor para contrastar hipótesis, cuya validez fue siempre verificada por los autores.

En ningún caso la inteligencia artificial ha sustituido el criterio técnico o académico del equipo. Los autores asumen plena responsabilidad sobre el contenido, los análisis y las conclusiones de este trabajo.